In [195]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

from combine_features import read_data

# Experiments

In [69]:
df = read_data("../rhea-soil-nutrient-prediction-challenge/Train.csv")
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Cu,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,5.826,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,4.346,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.657,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.376,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.351,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN


In [70]:
df.iloc[:, 40:].info()

<class 'pandas.DataFrame'>
RangeIndex: 13454 entries, 0 to 13453
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ca         13454 non-null  float64
 1   C_organic  13454 non-null  float64
 2   C_total    13416 non-null  float64
 3   Cu         13416 non-null  float64
 4   Fe         13416 non-null  float64
 5   Mg         13454 non-null  float64
 6   Mn         13416 non-null  float64
 7   N          13416 non-null  float64
 8   P          0 non-null      float64
 9   K          13454 non-null  float64
 10  Na         38 non-null     float64
 11  S          0 non-null      float64
 12  Zn         0 non-null      float64
dtypes: float64(13)
memory usage: 1.3 MB


In [71]:
pred_to_keep = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TargetPred_To_Keep.csv")
pred_to_keep.head()

,ID,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [72]:
pred_to_keep.sum()

ID    00A83S00F2Q300FNFP01MFSS02851F02C3Q502I9O502L2...
Al                                                 6070
B                                                  1345
Ca                                                 6070
Cu                                                 6065
Fe                                                 6065
K                                                  6070
Mg                                                 6070
Mn                                                 6065
N                                                  6065
Na                                                 1350
P                                                  1345
S                                                  1345
Zn                                                 1345
dtype: object

In [73]:
encoder = LabelEncoder()
df['Depth'] = encoder.fit_transform(df['Depth_cm'])
# df.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True)
# df.dropna(axis=1, thresh=int(0.8 * df.shape[0]), inplace=True) # drops target columns :(
# df.dropna(axis=0, inplace=True)
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Fe,Mg,Mn,N,P,K,Na,S,Zn,Depth
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN,1
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN,1
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN,1
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN,1
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN,1


In [74]:
target = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "K",
        "Mg",
        "Mn",
        "N",
        "Na",
        "P",
        "S",
        "Zn",
    ]
x = df.drop(columns=target, errors='ignore')
x = x.drop(columns=['ID', 'Area', 'Depth_cm', 'ph'], errors='ignore')
y_columns = df.columns.difference(x.columns)
y_columns = [col for col in y_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y = df[y_columns]
y.fillna(0, inplace=True)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [75]:
x_train.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,"Beans, dry",...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
4721,-2.19999,7.20972,-2.917073,-2.796745,-18.097209,22.215910,30.598484,103.590550,10390.800000,1165.118182,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
7732,-11.01100,14.76583,3.931445,-2.550627,-3.758418,23.316288,37.174244,54.082947,34266.136364,811.044444,...,3123.272727,985.490909,420.227273,1037.981818,620.363636,72963.909091,17452.418182,17080.290909,2304.281818,0
5746,-1.07758,7.64100,-2.917073,-2.796745,-18.097209,22.660984,32.393940,103.886536,10390.800000,1165.118182,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
12178,14.92311,-16.19789,9.219964,0.017264,-9.459173,15.484848,30.342804,61.349070,24064.809091,405.136364,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0
12952,15.22342,-12.29724,9.219964,0.017264,-9.459173,13.121212,25.611742,121.739590,24064.809091,405.136364,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0


In [76]:
y_train.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn,ph
4721,668.664,0.0,2440.023,3.406,141.737,138.646,286.859,337.732,1.90,0.0,0.0,0.0,0.0,6.969
7732,981.937,0.0,2336.159,2.910,159.714,271.761,671.278,93.069,0.56,0.0,0.0,0.0,0.0,6.699
5746,837.742,0.0,2120.437,2.700,109.035,218.283,327.699,250.103,1.91,0.0,0.0,0.0,0.0,6.509
12178,730.772,0.0,740.481,0.966,172.359,87.640,134.619,102.362,0.80,0.0,0.0,0.0,0.0,6.161
12952,1145.199,0.0,541.405,2.794,83.008,160.257,149.018,69.130,0.62,0.0,0.0,0.0,0.0,5.903


In [77]:
models = {}
for col in y_columns:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    models[col] = model
    print(f"{col}: RMSE = {rmse}")

Al: RMSE = 130.2396468091458
B: RMSE = 0.0
Ca: RMSE = 959.4224269660466
Cu: RMSE = 0.7452205663210932
Fe: RMSE = 29.148475487969623
K: RMSE = 100.47139795477366
Mg: RMSE = 134.07210263822418
Mn: RMSE = 43.330061762577444
N: RMSE = 0.46218505583420955
Na: RMSE = 1.4646935033365307
P: RMSE = 0.0
S: RMSE = 0.0
Zn: RMSE = 0.0
ph: RMSE = 0.34927910272145685


In [78]:
for col, model in models.items():
    fi = pd.DataFrame(model.feature_importances_, index=x_train.columns, columns=['Feature Importance'])
    fi.sort_values(by='Feature Importance', ascending=False, inplace=True)
    print(f"Feature importance for {col}:")
    print(fi.head(10))

Feature importance for Al:
                                   Feature Importance
prec_avg                                     0.536625
Longitude                                    0.152047
Latitude                                     0.119135
tmin_avg                                     0.052261
Soya beans                                   0.046809
tmax_avg                                     0.022330
Pineapples                                   0.020283
Depth                                        0.012970
Coffee, green                                0.005833
Cropland phosphorus per unit area            0.004214
Feature importance for B:
                                                 Feature Importance
Longitude                                                       0.0
Seed cotton, unginned                                           0.0
Other fruits, n.e.c.                                            0.0
Other pulses n.e.c.                                             0.0
Other vegetab

# Generating test answers

## Importing df

In [264]:
train = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/Train.csv")
train.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm,Al,B,...,Fe,Mg,Mn,N,ph,P,K,Na,S,Zn
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50,1109.856,NaN,...,92.366,200.601,107.257,2.24,5.942,NaN,283.103,NaN,NaN,NaN
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50,1168.364,NaN,...,115.923,197.771,90.005,1.57,5.722,NaN,215.459,NaN,NaN,NaN
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50,1137.113,NaN,...,78.709,188.114,120.433,1.02,5.510,NaN,398.656,NaN,NaN,NaN
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50,1117.349,NaN,...,127.527,156.417,112.036,1.12,5.817,NaN,267.354,NaN,NaN,NaN
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50,1219.203,NaN,...,77.542,114.809,57.906,1.19,4.980,NaN,229.682,NaN,NaN,NaN


In [265]:
test_df = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TestSet.csv")
test_df.head()

,ID,Latitude,Longitude,Depth_cm,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,-0.746,37.094,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DCC6DM,-0.785,37.178,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,T50LK1,-0.629,37.126,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FNLYT0,-0.351,35.308,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FP5E12,-1.894,36.987,0-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [266]:
pred_to_keep = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TargetPred_To_Keep.csv")
pred_to_keep.head()

,ID,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [267]:
olek_features = pd.read_csv("../features/olek-features.csv")
print(olek_features.shape)
olek_features.head()

(50368, 59)


,ID,Country,Area,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Fe_pred,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,O2TONM,Tanzania,Tanzania,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,Tanzania,Tanzania,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,Tanzania,Tanzania,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,Tanzania,Tanzania,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,Tanzania,Tanzania,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


## Combining data

In [268]:
features = olek_features.copy()
features.drop(columns=['Country', 'Area'], inplace=True)
features.head()

,ID,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,B4,B8,...,Fe_pred,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,O2TONM,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,276.0,1782.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,1140.0,2714.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,1114.0,2484.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,750.0,2032.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,377.0,2620.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [269]:
train = pd.merge(train, features, on='ID', how='left')
train.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm,Al,B,...,Fe_pred,K_pred,Mg_pred,N_y,P_pred,S_pred,Zn_pred,carbon,clay,ph_y
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50,1109.856,NaN,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50,1168.364,NaN,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50,1137.113,NaN,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50,1117.349,NaN,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50,1219.203,NaN,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [270]:
test = pd.merge(test_df, features, on='ID', how='left')
test.head()

,ID,Latitude,Longitude,Depth_cm,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,...,Fe_pred,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,8ZMJRO,-0.746,37.094,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,65.686331,329.299560,402.428793,2.434201e+07,10.023176,8.025013,8.025013,38.0,42.0,5.9
1,DCC6DM,-0.785,37.178,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,72.699794,163.021907,220.406416,8.954293e+06,12.463738,7.166170,4.473947,35.0,43.0,5.6
2,T50LK1,-0.629,37.126,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,89.017131,163.021907,220.406416,5.987314e+07,12.463738,8.974182,3.953032,33.0,37.0,5.6
3,FNLYT0,-0.351,35.308,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,98.484316,243.691932,180.272242,5.403639e+08,12.463738,3.481689,3.953032,36.0,39.0,5.5
4,FP5E12,-1.894,36.987,0-20,0.0,0.0,0.0,0.0,0.0,0.0,...,72.699794,243.691932,269.426407,8.114058e+05,17.174145,3.953032,1.718282,28.0,29.0,6.4


In [271]:
encoder = LabelEncoder()
train['Depth'] = encoder.fit_transform(train['Depth_cm'])
test['Depth'] = encoder.transform(test['Depth_cm'])
train.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True, errors='ignore')
test.drop(columns=['ID', 'Depth_cm'], inplace=True, errors='ignore')
test.head()

,Latitude,Longitude,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,...,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph,Depth
0,-0.746,37.094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,329.299560,402.428793,2.434201e+07,10.023176,8.025013,8.025013,38.0,42.0,5.9,0
1,-0.785,37.178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,163.021907,220.406416,8.954293e+06,12.463738,7.166170,4.473947,35.0,43.0,5.6,0
2,-0.629,37.126,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,163.021907,220.406416,5.987314e+07,12.463738,8.974182,3.953032,33.0,37.0,5.6,0
3,-0.351,35.308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,243.691932,180.272242,5.403639e+08,12.463738,3.481689,3.953032,36.0,39.0,5.5,0
4,-1.894,36.987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,243.691932,269.426407,8.114058e+05,17.174145,3.953032,1.718282,28.0,29.0,6.4,0


In [272]:
train["Na"] = train["Na"].fillna(train["Na"].mean(), inplace=False)
train = train.fillna(0, inplace=False)
train.head()

,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Al,B,Ca,C_organic,...,K_pred,Mg_pred,N_y,P_pred,S_pred,Zn_pred,carbon,clay,ph_y,Depth
0,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,1109.856,0.0,1535.388,30.66,...,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6,1
1,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,1168.364,0.0,751.408,21.15,...,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7,1
2,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,1137.113,0.0,468.391,15.64,...,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7,1
3,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,1117.349,0.0,739.698,15.63,...,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5,1
4,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,1219.203,0.0,240.071,18.49,...,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0,1


## Separating Data

In [273]:
target = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "K",
        "Mg",
        "Mn",
        "N",
        "Na",
        "P",
        "S",
        "Zn",
    ]
X = train.drop(columns=target, errors='ignore')
y_columns = train.columns.difference(X.columns)
y_columns = [col for col in y_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y = train[y_columns]


In [274]:
X.head()

,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,electrical_conductivity,N_x,ph_x,tmin_avg,...,K_pred,Mg_pred,N_y,P_pred,S_pred,Zn_pred,carbon,clay,ph_y,Depth
0,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,0.0,2.24,5.942,12.727273,...,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6,1
1,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,0.0,1.57,5.722,12.727273,...,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7,1
2,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,0.0,1.02,5.510,12.727273,...,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7,1
3,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,0.0,1.12,5.817,12.727273,...,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5,1
4,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,0.0,1.19,4.980,12.674242,...,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0,1


In [277]:
X.drop(columns=['ph_x'], inplace=True)
X.head()

,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,electrical_conductivity,N_x,tmin_avg,tmax_avg,...,K_pred,Mg_pred,N_y,P_pred,S_pred,Zn_pred,carbon,clay,ph_y,Depth
0,0.604867,-0.463864,01/01/2008,31/12/2018,1.091238,1.091238,-0.199663,1.657001,-1.201331,-1.831983,...,-0.278184,-0.610901,-0.021464,0.480508,-0.604010,-0.890036,1.012217,1.245357,-0.880190,1.091238
1,0.604732,-0.463769,01/01/2008,31/12/2018,1.091238,1.091238,-0.199663,0.778847,-1.201331,-1.831983,...,-0.278184,-0.652835,-0.022868,-0.240003,-0.687651,-1.127053,0.614484,1.767473,-0.745993,1.091238
2,0.604747,-0.463685,01/01/2008,31/12/2018,1.091238,1.091238,-0.199663,0.057975,-1.201331,-1.831983,...,-0.278184,-0.610901,-0.023076,-0.240003,-0.687651,-1.127053,0.614484,1.636944,-0.745993,1.091238
3,0.604533,-0.463639,01/01/2008,31/12/2018,1.091238,1.091238,-0.199663,0.189043,-1.201331,-1.831983,...,-0.443964,-0.564556,-0.022868,-0.240003,-0.763333,-0.890036,0.813350,1.767473,-1.014387,1.091238
4,0.600935,-0.463457,01/01/2008,31/12/2018,1.091238,1.091238,-0.199663,0.280790,-1.216034,-1.694780,...,-0.182000,-0.690779,-0.023020,-0.240003,-0.687651,-1.127053,1.211083,1.636944,-1.685372,1.091238


In [278]:
y.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,37.98841,0.0,0.0,0.0


In [279]:
test.head()

,Latitude,Longitude,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,...,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph,Depth
0,-0.746,37.094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,329.299560,402.428793,2.434201e+07,10.023176,8.025013,8.025013,38.0,42.0,5.9,0
1,-0.785,37.178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,163.021907,220.406416,8.954293e+06,12.463738,7.166170,4.473947,35.0,43.0,5.6,0
2,-0.629,37.126,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,163.021907,220.406416,5.987314e+07,12.463738,8.974182,3.953032,33.0,37.0,5.6,0
3,-0.351,35.308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,243.691932,180.272242,5.403639e+08,12.463738,3.481689,3.953032,36.0,39.0,5.5,0
4,-1.894,36.987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,243.691932,269.426407,8.114058e+05,17.174145,3.953032,1.718282,28.0,29.0,6.4,0


In [280]:
target_pred = [
    "Target_Al",
    "Target_B",
    "Target_Ca",
    "Target_Cu",
    "Target_Fe",
    "Target_K",
    "Target_Mg",
    "Target_Mn",
    "Target_N",
    "Target_Na",
    "Target_P",
    "Target_S",
    "Target_Zn",
]

In [281]:
X_pred = test.drop(columns=target_pred, errors='ignore')
y_pred_columns = test.columns.difference(X_pred.columns)
y_pred_columns = [col for col in y_pred_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y_pred = test[y_pred_columns]

In [282]:
X_pred.head()

,Latitude,Longitude,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,B4,...,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph,Depth
0,-0.746,37.094,14.543561,25.986742,108.980880,1715.000000,1098.000000,363.00,600.000000,680.00,...,329.299560,402.428793,2.434201e+07,10.023176,8.025013,8.025013,38.0,42.0,5.9,0
1,-0.785,37.178,15.577652,27.297348,91.997140,2444.000000,1377.000000,407.00,705.000000,607.00,...,163.021907,220.406416,8.954293e+06,12.463738,7.166170,4.473947,35.0,43.0,5.6,0
2,-0.629,37.126,14.102273,25.522728,115.549255,1933.000000,1084.000000,376.00,625.000000,633.00,...,163.021907,220.406416,5.987314e+07,12.463738,8.974182,3.953032,33.0,37.0,5.6,0
3,-0.351,35.308,11.918561,24.070076,165.598710,1998.833333,846.142857,287.65,738.588235,345.45,...,243.691932,180.272242,5.403639e+08,12.463738,3.481689,3.953032,36.0,39.0,5.5,0
4,-1.894,36.987,14.017045,25.181818,45.469700,3060.500000,2247.500000,499.00,750.500000,922.00,...,243.691932,269.426407,8.114058e+05,17.174145,3.953032,1.718282,28.0,29.0,6.4,0


In [283]:
y_pred.head()

,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [284]:
diff = set(X.columns) - set(X_pred.columns)
X.drop(columns=diff, inplace=True)

In [285]:
X_pred = X_pred[X.columns]
X_pred.head()

,Longitude,Latitude,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,B4,...,Ca_pred,Fe_pred,K_pred,Mg_pred,P_pred,S_pred,Zn_pred,carbon,clay,Depth
0,37.094,-0.746,14.543561,25.986742,108.980880,1715.000000,1098.000000,363.00,600.000000,680.00,...,1095.633158,65.686331,329.299560,402.428793,10.023176,8.025013,8.025013,38.0,42.0,0
1,37.178,-0.785,15.577652,27.297348,91.997140,2444.000000,1377.000000,407.00,705.000000,607.00,...,734.095189,72.699794,163.021907,220.406416,12.463738,7.166170,4.473947,35.0,43.0,0
2,37.126,-0.629,14.102273,25.522728,115.549255,1933.000000,1084.000000,376.00,625.000000,633.00,...,811.405825,89.017131,163.021907,220.406416,12.463738,8.974182,3.953032,33.0,37.0,0
3,35.308,-0.351,11.918561,24.070076,165.598710,1998.833333,846.142857,287.65,738.588235,345.45,...,896.847292,98.484316,243.691932,180.272242,12.463738,3.481689,3.953032,36.0,39.0,0
4,36.987,-1.894,14.017045,25.181818,45.469700,3060.500000,2247.500000,499.00,750.500000,922.00,...,1095.633158,72.699794,243.691932,269.426407,17.174145,3.953032,1.718282,28.0,29.0,0


## Standarization

In [286]:
scaler = StandardScaler()
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
X_pred[numeric_cols] = scaler.transform(X_pred[numeric_cols])

## Prognosing ph feature(currently unused)

In [68]:
ph = X['ph']
X = X.drop(columns=['ph'], errors='ignore')
X.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,37.65189,-3.15440,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,1477.0,1153.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
1,37.63612,-3.08585,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,1513.5,1084.5,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
2,39.55580,-2.67218,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2198.0,1346.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
3,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2258.0,1621.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1
4,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2258.0,1621.0,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1


In [69]:
X_pred = X_pred[X.columns]
X_pred.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
0,37.094,-0.746,14.202173,3.902236,-7.263155,14.543561,25.986742,108.980880,1715.000000,1098.000000,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
1,37.178,-0.785,14.202173,3.902236,-7.263155,15.577652,27.297348,91.997140,2444.000000,1377.000000,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
2,37.126,-0.629,14.202173,3.902236,-7.263155,14.102273,25.522728,115.549255,1933.000000,1084.000000,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
3,35.308,-0.351,14.202173,3.902236,-7.263155,11.918561,24.070076,165.598710,1998.833333,846.142857,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0
4,36.987,-1.894,14.202173,3.902236,-7.263155,14.017045,25.181818,45.469700,3060.500000,2247.500000,...,3498.736364,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0


In [70]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, ph)
ph_pred = model.predict(X_pred)
ph_pred[:10]

array([6.36927   , 6.24246667, 6.26201333, 6.57290667, 7.10632167,
       6.58553333, 6.182543  , 6.27559   , 7.180215  , 7.62547333])

In [71]:
len(ph), len(ph_pred), X.shape, X_pred.shape

(13454, 6070, (13454, 41), (6070, 41))

In [72]:
X = pd.concat([X, ph], axis=1)
X.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,...,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth,ph
0,37.65189,-3.15440,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,1477.0,1153.0,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,6.405
1,37.63612,-3.08585,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,1513.5,1084.5,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,6.419
2,39.55580,-2.67218,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2198.0,1346.0,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,8.388
3,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2258.0,1621.0,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,8.302
4,39.55477,-2.67196,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2258.0,1621.0,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,1,8.292


In [73]:
X_pred['ph'] = ph_pred
X_pred.head()

,Longitude,Latitude,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,B12,...,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth,ph
0,37.094,-0.746,14.202173,3.902236,-7.263155,14.543561,25.986742,108.980880,1715.000000,1098.000000,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.369270
1,37.178,-0.785,14.202173,3.902236,-7.263155,15.577652,27.297348,91.997140,2444.000000,1377.000000,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.242467
2,37.126,-0.629,14.202173,3.902236,-7.263155,14.102273,25.522728,115.549255,1933.000000,1084.000000,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.262013
3,35.308,-0.351,14.202173,3.902236,-7.263155,11.918561,24.070076,165.598710,1998.833333,846.142857,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,6.572907
4,36.987,-1.894,14.202173,3.902236,-7.263155,14.017045,25.181818,45.469700,3060.500000,2247.500000,...,500.454545,485.872727,715.863636,1038.663636,79866.045455,13190.418182,22006.490909,711.218182,0,7.106322


In [5]:
X.drop(columns=['ph'], inplace=True)
X.columns

NameError: name 'X' is not defined

## Additional prep for model

In [287]:
X.columns

Index(['Longitude', 'Latitude', 'tmin_avg', 'tmax_avg', 'prec_avg', 'B11_x',
       'B12_x', 'B2', 'B3', 'B4', 'B8', 'aspect', 'elevation', 'hillshade',
       'slope', 'cec_0-5cm_mean', 'cec_15-30cm_mean', 'cec_30-60cm_mean',
       'cec_5-15cm_mean', 'clay_0-5cm_mean', 'clay_15-30cm_mean',
       'clay_30-60cm_mean', 'clay_5-15cm_mean', 'phh2o_0-5cm_mean',
       'phh2o_15-30cm_mean', 'phh2o_30-60cm_mean', 'phh2o_5-15cm_mean',
       'sand_0-5cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean',
       'sand_5-15cm_mean', 'soc_0-5cm_mean', 'soc_15-30cm_mean',
       'soc_30-60cm_mean', 'soc_5-15cm_mean', 'B04', 'B05', 'B06', 'B07',
       'B08', 'B09', 'B10', 'B11_y', 'B12_y', 'B13', 'B14', 'Al_pred',
       'Ca_pred', 'Fe_pred', 'K_pred', 'Mg_pred', 'P_pred', 'S_pred',
       'Zn_pred', 'carbon', 'clay', 'Depth'],
      dtype='str')

In [288]:
features_to_drop = ['Longitude', 'Latitude', 'Cropland nitrogen per unit area',
       'Cropland phosphorus per unit area', 'Cropland potassium per unit area', 'Bananas', 'Beans, dry',
       'Cashew nuts, in shell', 'Cassava, fresh',
       'Chillies and peppers, green (Capsicum spp. and Pimenta spp.)',
       'Coffee, green', 'Groundnuts, excluding shelled', 'Maize (corn)',
       'Mangoes, guavas and mangosteens', 'Millet',
       'Onions and shallots, dry (excluding dehydrated)', 'Oranges',
       'Other fruits, n.e.c.', 'Other pulses n.e.c.',
       'Other vegetables, fresh n.e.c.', 'Pineapples', 'Potatoes', 'Rice',
       'Seed cotton, unginned', 'Sesame seed', 'Sorghum', 'Soya beans',
       'Sugar cane', 'Sweet potatoes', 'Tomatoes', 'Unmanufactured tobacco',
       'Depth']

In [289]:
X.drop(columns=features_to_drop, inplace=True, errors='ignore')
X_pred.drop(columns=features_to_drop, inplace=True, errors='ignore')

In [296]:
check = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/Train.csv")
check.head()

,ID,Longitude,Latitude,start_date,end_date,horizon_lower,horizon_upper,Depth_cm,Al,B,...,Fe,Mg,Mn,N,ph,P,K,Na,S,Zn
0,O2TONM,35.18756,-8.62390,01/01/2008,31/12/2018,50,20,20-50,1109.856,NaN,...,92.366,200.601,107.257,2.24,5.942,NaN,283.103,NaN,NaN,NaN
1,BQLUK6,35.18558,-8.62300,01/01/2008,31/12/2018,50,20,20-50,1168.364,NaN,...,115.923,197.771,90.005,1.57,5.722,NaN,215.459,NaN,NaN,NaN
2,LSET8M,35.18579,-8.62221,01/01/2008,31/12/2018,50,20,20-50,1137.113,NaN,...,78.709,188.114,120.433,1.02,5.510,NaN,398.656,NaN,NaN,NaN
3,LEEL7I,35.18266,-8.62177,01/01/2008,31/12/2018,50,20,20-50,1117.349,NaN,...,127.527,156.417,112.036,1.12,5.817,NaN,267.354,NaN,NaN,NaN
4,LDNGO2,35.12984,-8.62005,01/01/2008,31/12/2018,50,20,20-50,1219.203,NaN,...,77.542,114.809,57.906,1.19,4.980,NaN,229.682,NaN,NaN,NaN


In [298]:
check.iloc[:,5:].info()

<class 'pandas.DataFrame'>
RangeIndex: 44298 entries, 0 to 44297
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   horizon_lower            44298 non-null  int64  
 1   horizon_upper            44298 non-null  int64  
 2   Depth_cm                 44298 non-null  str    
 3   Al                       44296 non-null  float64
 4   B                        1909 non-null   float64
 5   Ca                       44298 non-null  float64
 6   C_organic                44298 non-null  float64
 7   C_total                  42348 non-null  float64
 8   Cu                       44257 non-null  float64
 9   electrical_conductivity  1907 non-null   float64
 10  Fe                       44257 non-null  float64
 11  Mg                       44298 non-null  float64
 12  Mn                       44255 non-null  float64
 13  N                        44257 non-null  float64
 14  ph                       44295 no

## Basic model

In [299]:
models = {}
for idx, col in enumerate(y_pred_columns):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y.iloc[:, idx])
    y_pred.iloc[:, idx] = model.predict(X_pred)
    models[col] = model
    print(col, y.iloc[:, idx])

Target_Al 0        1109.856
1        1168.364
2        1137.113
3        1117.349
4        1219.203
           ...   
44293    1056.900
44294     320.713
44295     452.637
44296       0.000
44297       0.000
Name: Al, Length: 44298, dtype: float64
Target_B 0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.000000
           ...   
44293    0.009823
44294    0.009902
44295    0.009937
44296    0.360000
44297    0.380000
Name: B, Length: 44298, dtype: float64
Target_Ca 0        1535.388
1         751.408
2         468.391
3         739.698
4         240.071
           ...   
44293     692.423
44294     475.510
44295     318.088
44296    1390.000
44297    1430.000
Name: Ca, Length: 44298, dtype: float64
Target_Cu 0        2.259000
1        1.822000
2        1.913000
3        2.876000
4        1.825000
           ...   
44293    1.162110
44294    0.095523
44295    0.462887
44296    2.220000
44297    2.430000
Name: Cu, Length: 44298, dtype: float64
Target_Fe 

IndexError: single positional indexer is out-of-bounds

In [301]:
print(y.shape)
y.head()

(44298, 12)


,Al,B,Ca,Cu,Fe,K,Mg,Mn,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,37.98841,0.0,0.0,0.0


In [302]:
print(y_pred.shape)
y_pred.head()

(6070, 13)


,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,1540.223985,0.208071,16724.079227,4.233323,293.22949,1864.09500,2159.528746,210.646136,75.776177,24.720507,3.226488,0.583553,0.0
1,1540.223985,0.208071,16711.674999,4.227703,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.720507,3.226488,0.583553,0.0
2,1540.223985,0.208071,16711.674999,4.213542,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.783627,3.226488,0.583553,0.0
3,1540.223985,0.208071,16686.128239,4.212302,293.36433,1978.61073,2192.241223,209.036006,74.463220,24.677201,3.262684,0.583553,0.0
4,1556.552625,0.208071,16686.128239,4.212302,293.36433,1978.61073,2195.959629,209.036006,77.643333,24.677201,3.262684,0.583553,0.0


In [303]:
olek_features.head()

,ID,Country,Area,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,...,Fe_pred,K_pred,Mg_pred,N,P_pred,S_pred,Zn_pred,carbon,clay,ph
0,O2TONM,Tanzania,Tanzania,12.727273,22.628788,123.26878,1287.0,699.0,243.0,364.0,...,28.964100,133.289780,120.510418,1.208738e+07,8.974182,2.320117,0.822119,32.0,36.0,5.6
1,BQLUK6,Tanzania,Tanzania,12.727273,22.628788,123.26878,3215.0,2426.0,789.0,1066.0,...,26.112639,133.289780,108.947172,2.979958e+06,7.166170,2.004166,0.648721,30.0,40.0,5.7
2,LSET8M,Tanzania,Tanzania,12.727273,22.628788,123.26878,3127.0,2220.0,697.0,943.0,...,28.964100,133.289780,120.510418,1.634984e+06,7.166170,2.004166,0.648721,30.0,39.0,5.7
3,LEEL7I,Tanzania,Tanzania,12.727273,22.628788,123.26878,2085.0,1370.0,499.0,691.0,...,26.112639,108.947172,133.289780,2.979958e+06,7.166170,1.718282,0.822119,31.0,40.0,5.5
4,LDNGO2,Tanzania,Tanzania,12.674242,23.077652,110.35551,2059.0,1076.0,324.0,436.0,...,23.532530,147.413159,98.484316,1.997196e+06,7.166170,2.004166,0.648721,33.0,39.0,5.0


In [306]:
columns = y_pred.columns
missing_minerals = olek_features[['ID', 'S_pred', 'Zn_pred', 'P_pred']]
y_pred.merge(missing_minerals, on='ID', how='left')
y_pred.head()

KeyError: 'ID'

In [307]:
id = test_df["ID"]
result = pd.concat([id, y_pred], axis=1)
result.head()


,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1540.223985,0.208071,16724.079227,4.233323,293.22949,1864.09500,2159.528746,210.646136,75.776177,24.720507,3.226488,0.583553,0.0
1,DCC6DM,1540.223985,0.208071,16711.674999,4.227703,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.720507,3.226488,0.583553,0.0
2,T50LK1,1540.223985,0.208071,16711.674999,4.213542,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.783627,3.226488,0.583553,0.0
3,FNLYT0,1540.223985,0.208071,16686.128239,4.212302,293.36433,1978.61073,2192.241223,209.036006,74.463220,24.677201,3.262684,0.583553,0.0
4,FP5E12,1556.552625,0.208071,16686.128239,4.212302,293.36433,1978.61073,2195.959629,209.036006,77.643333,24.677201,3.262684,0.583553,0.0


In [308]:
pred_to_keep.shape

(6070, 14)

In [309]:
pred_to_keep.columns = result.columns
pred_to_keep.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [310]:
result_indexed = result.set_index("ID")
pred_indexed = pred_to_keep.set_index("ID")

cols = pred_indexed.columns
result_indexed[cols] = result_indexed[cols].where(pred_indexed[cols] == 1, other=0)

result = result_indexed.reset_index()
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1540.223985,0.208071,16724.079227,4.233323,293.22949,1864.09500,2159.528746,210.646136,75.776177,24.720507,3.226488,0.583553,0.0
1,DCC6DM,1540.223985,0.208071,16711.674999,4.227703,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.720507,3.226488,0.583553,0.0
2,T50LK1,1540.223985,0.208071,16711.674999,4.213542,293.22949,2031.40577,2160.196873,210.237806,77.961780,24.783627,3.226488,0.583553,0.0
3,FNLYT0,1540.223985,0.208071,16686.128239,4.212302,293.36433,1978.61073,2192.241223,209.036006,74.463220,24.677201,3.262684,0.583553,0.0
4,FP5E12,1556.552625,0.208071,16686.128239,4.212302,293.36433,1978.61073,2195.959629,209.036006,77.643333,24.677201,3.262684,0.583553,0.0


In [311]:
result.to_csv("../rhea-soil-nutrient-prediction-challenge/submissions/first-minerals-submission.csv", index=False)

## XGB

In [312]:
X.head()

,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,B4,B8,aspect,...,Al_pred,Ca_pred,Fe_pred,K_pred,Mg_pred,P_pred,S_pred,Zn_pred,carbon,clay
0,-1.201331,-1.831983,1.410737,-2.209292,-1.963817,-1.845951,-1.952308,-1.808011,-1.662627,0.252200,...,0.574404,-0.409989,-0.787508,-0.278184,-0.610901,0.480508,-0.604010,-0.890036,1.012217,1.245357
1,-1.201331,-1.831983,1.410737,0.177545,0.208481,0.595568,0.446016,-0.090097,0.419462,0.934989,...,0.574404,-0.438085,-0.889023,-0.278184,-0.652835,-0.240003,-0.687651,-1.127053,0.614484,1.767473
2,-1.201331,-1.831983,1.410737,0.068602,-0.050635,0.184177,0.025797,-0.141793,-0.094358,1.402652,...,0.308861,-0.463506,-0.787508,-0.278184,-0.610901,-0.240003,-0.687651,-1.127053,0.614484,1.636944
3,-1.201331,-1.831983,1.410737,-1.221379,-1.119803,-0.701209,-0.835140,-0.865544,-1.104127,-1.122731,...,0.574404,-0.507322,-0.889023,-0.443964,-0.564556,-0.240003,-0.763333,-0.890036,0.813350,1.767473
4,-1.216034,-1.694780,0.963239,-1.253567,-1.489610,-1.483747,-1.706326,-1.607190,0.209466,1.056581,...,0.574404,-0.558614,-0.980878,-0.182000,-0.690779,-0.240003,-0.687651,-1.127053,1.211083,1.636944


In [313]:
X_pred.head()

,tmin_avg,tmax_avg,prec_avg,B11_x,B12_x,B2,B3,B4,B8,aspect,...,Al_pred,Ca_pred,Fe_pred,K_pred,Mg_pred,P_pred,S_pred,Zn_pred,carbon,clay
0,14.543561,25.986742,108.980880,1715.000000,1098.000000,363.00,600.000000,680.00,2621.000000,63.0,...,329.299560,1095.633158,65.686331,329.299560,402.428793,10.023176,8.025013,8.025013,38.0,42.0
1,15.577652,27.297348,91.997140,2444.000000,1377.000000,407.00,705.000000,607.00,3306.000000,39.0,...,220.406416,734.095189,72.699794,163.021907,220.406416,12.463738,7.166170,4.473947,35.0,43.0
2,14.102273,25.522728,115.549255,1933.000000,1084.000000,376.00,625.000000,633.00,2854.000000,298.0,...,329.299560,811.405825,89.017131,163.021907,220.406416,12.463738,8.974182,3.953032,33.0,37.0
3,11.918561,24.070076,165.598710,1998.833333,846.142857,287.65,738.588235,345.45,4579.285714,173.0,...,269.426407,896.847292,98.484316,243.691932,180.272242,12.463738,3.481689,3.953032,36.0,39.0
4,14.017045,25.181818,45.469700,3060.500000,2247.500000,499.00,750.500000,922.00,2393.000000,315.0,...,147.413159,1095.633158,72.699794,243.691932,269.426407,17.174145,3.953032,1.718282,28.0,29.0


In [314]:
y.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,37.98841,0.0,0.0,0.0


In [ ]:
models = {}
for idx, col in enumerate(y_pred_columns):
    model = XGBRegressor(n_estimators=1000, random_state=42)
    model.fit(X, y.iloc[:, idx], verbose=False)
    y_pred.iloc[:, idx] = model.predict(X_pred)
    models[col] = model
    print(col, y.iloc[:, idx])

Target_Al 0        1108.121
1        1102.400
2         699.279
3         905.183
4         703.832
           ...   
13449     349.000
13450    1650.000
13451     861.000
13452     170.000
13453     115.000
Name: Al, Length: 13454, dtype: float64
Target_B 0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
13449    0.0
13450    0.0
13451    0.0
13452    0.0
13453    0.0
Name: B, Length: 13454, dtype: float64
Target_Ca 0        1510.802
1        1592.621
2        7541.051
3        5296.174
4        5922.011
           ...   
13449     133.600
13450     174.000
13451     105.800
13452     352.000
13453     406.000
Name: Ca, Length: 13454, dtype: float64
Target_Cu 0        5.826
1        4.346
2        3.657
3        3.376
4        3.351
         ...  
13449    0.000
13450    0.000
13451    0.000
13452    0.000
13453    0.000
Name: Cu, Length: 13454, dtype: float64
Target_Fe 0        81.780
1        97.198
2        42.672
3        52.861
4        46.057
        

In [143]:
print(y_pred.shape)
y_pred.head()

(6070, 13)


,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,1142.374634,0.0,2812.468506,3.870714,159.949005,165.705307,632.682739,224.051865,2.644720,32.129364,0.0,0.0,0.0
1,896.380066,0.0,2844.757812,3.326280,185.121429,376.345825,657.277771,170.576111,1.793384,32.158688,0.0,0.0,0.0
2,1268.962646,0.0,2728.878174,2.330326,166.559326,316.379700,737.470642,221.666031,3.308956,31.685741,0.0,0.0,0.0
3,1337.523315,0.0,2620.776855,3.228681,178.334564,576.343567,832.497559,226.330872,2.938833,31.925537,0.0,0.0,0.0
4,595.373230,0.0,3361.533447,4.042684,112.238197,620.369995,856.571045,151.752731,0.887873,32.148026,0.0,0.0,0.0


In [144]:
id = test_df["ID"]
result = pd.concat([id, y_pred], axis=1)
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1142.374634,0.0,2812.468506,3.870714,159.949005,165.705307,632.682739,224.051865,2.644720,32.129364,0.0,0.0,0.0
1,DCC6DM,896.380066,0.0,2844.757812,3.326280,185.121429,376.345825,657.277771,170.576111,1.793384,32.158688,0.0,0.0,0.0
2,T50LK1,1268.962646,0.0,2728.878174,2.330326,166.559326,316.379700,737.470642,221.666031,3.308956,31.685741,0.0,0.0,0.0
3,FNLYT0,1337.523315,0.0,2620.776855,3.228681,178.334564,576.343567,832.497559,226.330872,2.938833,31.925537,0.0,0.0,0.0
4,FP5E12,595.373230,0.0,3361.533447,4.042684,112.238197,620.369995,856.571045,151.752731,0.887873,32.148026,0.0,0.0,0.0


In [145]:
pred_to_keep.columns = result.columns
pred_to_keep.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [146]:
result_indexed = result.set_index("ID")
pred_indexed = pred_to_keep.set_index("ID")

cols = pred_indexed.columns
result_indexed[cols] = result_indexed[cols].where(pred_indexed[cols] == 1, other=0)

result = result_indexed.reset_index()
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1142.374634,0.0,2812.468506,3.870714,159.949005,165.705307,632.682739,224.051865,2.644720,32.129364,0.0,0.0,0.0
1,DCC6DM,896.380066,0.0,2844.757812,3.326280,185.121429,376.345825,657.277771,170.576111,1.793384,32.158688,0.0,0.0,0.0
2,T50LK1,1268.962646,0.0,2728.878174,2.330326,166.559326,316.379700,737.470642,221.666031,3.308956,31.685741,0.0,0.0,0.0
3,FNLYT0,1337.523315,0.0,2620.776855,3.228681,178.334564,576.343567,832.497559,226.330872,2.938833,31.925537,0.0,0.0,0.0
4,FP5E12,595.373230,0.0,3361.533447,4.042684,112.238197,620.369995,856.571045,151.752731,0.887873,32.148026,0.0,0.0,0.0


In [147]:
result.to_csv("../rhea-soil-nutrient-prediction-challenge/submissions/first-XGBoost-submission.csv", index=False)

In [316]:
targets = y_columns 

models = {}
oof_preds = pd.DataFrame(index=X.index, columns=targets)
test_preds = pd.DataFrame(index=X_pred.index, columns=targets)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for target_name in targets:
    print(f"--- Training model for: {target_name} ---")
    y_target = y[target_name]
    
    if y_target.nunique() <= 1:
        print(f"{target_name} constant value. Returning 0.")
        oof_preds[target_name] = 0
        test_preds[target_name] = 0
        continue

    target_test_preds = np.zeros(len(X_pred))
    fold_scores = []
    
    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y_target)):
        X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
        y_train_fold, y_valid_fold = y_target.iloc[train_idx], y_target.iloc[valid_idx]
        
        model = XGBRegressor(
            n_estimators=2000,        
            learning_rate=0.03,       
            max_depth=5,              
            subsample=0.8,            
            colsample_bytree=0.8,     
            random_state=42 + fold,
            early_stopping_rounds=50, 
            n_jobs=-1
        )
        
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_valid_fold, y_valid_fold)],
            verbose=False
        )
        
        valid_pred = model.predict(X_valid_fold)
        oof_preds.loc[valid_idx, target_name] = valid_pred
        
        rmse = root_mean_squared_error(y_valid_fold, valid_pred)
        fold_scores.append(rmse)
        
        target_test_preds += model.predict(X_pred) / kf.n_splits
        
    print(f"Average RMSE for {target_name}: {np.mean(fold_scores):.4f}")
    
    test_preds[target_name] = target_test_preds

result = pd.DataFrame({'ID': test_df['ID']})
test_preds.columns = [f"Target_{col}" for col in test_preds.columns]
result = pd.concat([result, test_preds], axis=1)


--- Training model for: Al ---
Average RMSE for Al: 128.5460
--- Training model for: B ---
Average RMSE for B: 0.0127
--- Training model for: Ca ---
Average RMSE for Ca: 827.7355
--- Training model for: Cu ---
Average RMSE for Cu: 0.7658
--- Training model for: Fe ---
Average RMSE for Fe: 24.1403
--- Training model for: K ---
Average RMSE for K: 103.4242
--- Training model for: Mg ---
Average RMSE for Mg: 123.4850
--- Training model for: Mn ---
Average RMSE for Mn: 42.4963
--- Training model for: Na ---
Average RMSE for Na: 3.5066
--- Training model for: P ---
Average RMSE for P: 0.7269
--- Training model for: S ---
Average RMSE for S: 0.6011
--- Training model for: Zn ---
Average RMSE for Zn: 0.0745


In [318]:
y.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,Na,P,S,Zn
0,1109.856,0.0,1535.388,2.259,92.366,283.103,200.601,107.257,37.98841,0.0,0.0,0.0
1,1168.364,0.0,751.408,1.822,115.923,215.459,197.771,90.005,37.98841,0.0,0.0,0.0
2,1137.113,0.0,468.391,1.913,78.709,398.656,188.114,120.433,37.98841,0.0,0.0,0.0
3,1117.349,0.0,739.698,2.876,127.527,267.354,156.417,112.036,37.98841,0.0,0.0,0.0
4,1219.203,0.0,240.071,1.825,77.542,229.682,114.809,57.906,37.98841,0.0,0.0,0.0


In [317]:
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1942.213715,0.303463,17488.069824,4.504663,358.981415,2034.840546,2536.654480,297.935078,53.323477,18.873214,13.548475,1.312220
1,DCC6DM,2052.957306,0.304155,17486.200195,4.453849,365.809639,2020.965607,2410.561035,301.652763,82.400393,20.299960,13.625194,1.282585
2,T50LK1,2041.368591,0.304018,17592.353271,4.416485,364.927658,2020.965607,2391.607178,301.680794,82.191326,20.322557,13.570373,1.282472
3,FNLYT0,2080.748749,0.313428,17259.927979,4.668494,373.523643,1994.397278,2427.115814,303.157360,82.170771,20.322792,13.621921,1.301330
4,FP5E12,2045.894379,0.313428,17017.267334,4.770430,365.234200,1992.234955,2398.988586,303.801357,82.276464,20.336988,13.620596,1.300901


In [172]:
result_indexed = result.set_index("ID")
pred_indexed = pred_to_keep.set_index("ID")

cols = pred_indexed.columns
result_indexed[cols] = result_indexed[cols].where(pred_indexed[cols] == 1, other=0)

result = result_indexed.reset_index()
result.head()

,ID,Target_Al,Target_B,Target_Ca,Target_Cu,Target_Fe,Target_K,Target_Mg,Target_Mn,Target_N,Target_Na,Target_P,Target_S,Target_Zn
0,8ZMJRO,1091.049805,0,2404.258209,4.154161,144.076632,381.791389,689.747139,192.627197,1.922579,32.150399,0,0,0
1,DCC6DM,907.985092,0,2182.793701,3.316833,132.467848,458.362061,598.327278,161.574379,1.728570,32.149765,0,0,0
2,T50LK1,1125.768066,0,2866.016235,3.720158,151.386286,576.959740,895.657608,160.617167,2.192748,34.953802,0,0,0
3,FNLYT0,1387.627350,0,2558.254852,3.544839,137.268866,586.829720,483.582199,137.219517,3.033383,32.148636,0,0,0
4,FP5E12,638.226784,0,2997.869995,3.825842,106.462793,373.519569,670.825333,155.616892,1.010243,32.150007,0,0,0


In [173]:
result.to_csv("../rhea-soil-nutrient-prediction-challenge/submissions/removed-fao-fe-XGBoost-submission.csv", index=False)